# Haptic Signal VAE - Train, Freeze, and Export

Minimal Colab workflow for the first stage only:

1. Train the VAE using the same setup and config logic as `train_colab.ipynb`.
2. Freeze the trained VAE checkpoint as the source model for later analysis.
3. Extract deterministic latent `mu`, fit PCA, and save PCA artifacts.
4. Build the basic controllable PC specification.
5. Copy the model, PCA, and controls artifacts to Google Drive.

This notebook intentionally omits extended validation, AE baseline, cross-seed comparison, decode demos, and visualization galleries.

In [ ]:
# 1. Clone code repo (or force-update to latest)
import os

REPO_URL = "https://github.com/cindy-77jiayi/thesis_hapticAE.git"
REPO_DIR = "/content/thesis_hapticAE"

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
# 2. Clone dataset
DATASET_URL = "https://github.com/HapticGen/hapticgen-dataset.git"
DATASET_DIR = "/content/hapticgen-dataset"

if os.path.exists(DATASET_DIR):
    !cd {DATASET_DIR} && git pull
else:
    !git clone {DATASET_URL} {DATASET_DIR}

DATA_DIR = DATASET_DIR  # scan both expertvoted/ and uservoted/
print(f"Dataset root: {DATA_DIR}")
for sub in ["expertvoted", "uservoted"]:
    p = os.path.join(DATA_DIR, sub)
    n = len(os.listdir(p)) if os.path.exists(p) else 0
    print(f"  {sub}: {n} files")

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Configure run profile (aligned with current repo configs)
from pathlib import Path
import os

OUTPUT_DIR = "/content/outputs"

# Options: "baseline_balanced", "baseline_mixed"
RUN_PROFILE = "baseline_balanced"

# Set >1 if you want to train several candidates and keep only the best one.
NUM_TRAINING_RUNS = 1
CANDIDATE_EVAL_N_SAMPLES = 30
PCA_SELECTION_COMPONENTS = 8
STD_RATIO_TARGET = 1.0

# Lower quality_score is better. These are intentionally simple first-pass weights.
SELECTION_WEIGHTS = {
    'val_loss': 0.35,
    'mse': 0.20,
    'spectral_log_l1': 0.15,
    'std_ratio_error': 0.15,
    'pca_uncovered': 0.15,
}

PROFILE_TO_CONFIG = {
    "baseline_balanced": "configs/vae_balanced.yaml",
    "baseline_mixed": "configs/vae_balanced_mixed.yaml",
}

assert RUN_PROFILE in PROFILE_TO_CONFIG, f"Unknown RUN_PROFILE: {RUN_PROFILE}"
CONFIG = PROFILE_TO_CONFIG[RUN_PROFILE]
DATA_DIR = DATASET_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Profile: {RUN_PROFILE}")
print(f"Training attempts: {NUM_TRAINING_RUNS}")
print(f"Data:    {DATA_DIR}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Config:  {CONFIG}")
print(f"Selection weights: {SELECTION_WEIGHTS}")

In [ ]:
# 5. Run training candidates and select the best validation loss + reconstruction + PCA quality
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import numpy as np
import torch
import yaml

REPO_DIR = "/content/thesis_hapticAE"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.eval.evaluate import evaluate_reconstruction
from src.pipelines.latent_extraction import extract_latent_vectors
from src.pipelines.pca_control import fit_pca_pipeline


def compute_quality_score(m):
    """Lower is better. Uses log1p for loss-like terms so one metric does not dominate."""
    std_ratio_error = abs(float(m['std_ratio_mean']) - STD_RATIO_TARGET)
    pca_uncovered = max(0.0, 1.0 - float(m['pca_coverage']))
    components = {
        'val_loss': np.log1p(float(m['best_val_loss'])),
        'mse': np.log1p(float(m['mse_mean'])),
        'spectral_log_l1': np.log1p(float(m['spectral_log_l1_mean'])),
        'std_ratio_error': std_ratio_error,
        'pca_uncovered': pca_uncovered,
    }
    score = sum(SELECTION_WEIGHTS[k] * components[k] for k in SELECTION_WEIGHTS)
    return float(score), {k: float(v) for k, v in components.items()}

config_for_name = load_config(CONFIG)
run_name = config_for_name.get('run_name', 'vae_default')
set_seed(config_for_name.get('seed', 42))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_seed = int(config_for_name.get('seed', 42))
canonical_run_dir = Path(OUTPUT_DIR) / run_name
candidate_root = Path(OUTPUT_DIR) / '_training_candidates' / run_name
candidate_root.mkdir(parents=True, exist_ok=True)

# Same evaluation split and PCA dataset for every candidate.
candidate_eval_data = build_dataloaders(config_for_name, DATA_DIR, batch_size=max(32, CANDIDATE_EVAL_N_SAMPLES))
candidate_val_loader = candidate_eval_data['val_loader']
candidate_pca_data = build_dataloaders(config_for_name, DATA_DIR, batch_size=64, full_dataset=True)

candidate_reports = []
for attempt in range(1, NUM_TRAINING_RUNS + 1):
    attempt_output_dir = candidate_root / f'attempt_{attempt:02d}'
    attempt_output_dir.mkdir(parents=True, exist_ok=True)
    attempt_seed = base_seed + attempt - 1
    attempt_config = dict(config_for_name)
    attempt_config['seed'] = attempt_seed
    attempt_config_path = attempt_output_dir / 'attempt_config.yaml'
    with open(attempt_config_path, 'w') as f:
        yaml.safe_dump(attempt_config, f, sort_keys=False)
    print(f'\n=== Training candidate {attempt}/{NUM_TRAINING_RUNS}: {attempt_output_dir} | seed={attempt_seed} ===')
    subprocess.run([
        sys.executable, 'scripts/train.py',
        '--config', str(attempt_config_path),
        '--data_dir', DATA_DIR,
        '--output_dir', str(attempt_output_dir),
    ], check=True)

    attempt_run_dir = attempt_output_dir / run_name
    metrics_path_candidate = attempt_run_dir / 'metrics.npz'
    ckpt_candidate = attempt_run_dir / 'best_model.pt'
    assert ckpt_candidate.exists(), f'Missing checkpoint: {ckpt_candidate}'
    assert metrics_path_candidate.exists(), f'Missing metrics: {metrics_path_candidate}'

    metrics_candidate = np.load(metrics_path_candidate)
    best_val_loss = float(np.min(metrics_candidate['val_losses']))

    candidate_model = build_model(config_for_name, device)
    load_checkpoint(candidate_model, str(ckpt_candidate), device)
    candidate_model.eval()

    eval_result = evaluate_reconstruction(
        candidate_model,
        candidate_val_loader,
        device,
        n_samples=CANDIDATE_EVAL_N_SAMPLES,
        sr=config_for_name['data']['sr'],
    )
    std_ratio_mean = float(np.mean([r['std_ratio'] for r in eval_result['per_sample']]))
    mse_mean = float(eval_result['reconstruction_summary']['mse_mean'])
    spectral_log_l1_mean = float(eval_result['reconstruction_summary']['spectral_log_l1_mean'])

    Z_candidate = extract_latent_vectors(candidate_model, candidate_pca_data['all_loader'], device).astype(np.float32)
    pipe_candidate, _ = fit_pca_pipeline(Z_candidate, n_components=PCA_SELECTION_COMPONENTS)
    pca_coverage = float(pipe_candidate.named_steps['pca'].explained_variance_ratio_.sum())

    quality_metrics = {
        'best_val_loss': best_val_loss,
        'std_ratio_mean': std_ratio_mean,
        'mse_mean': mse_mean,
        'spectral_log_l1_mean': spectral_log_l1_mean,
        'pca_coverage': pca_coverage,
    }
    quality_score, quality_components = compute_quality_score(quality_metrics)

    candidate_reports.append({
        'attempt': attempt,
        'seed': attempt_seed,
        'config': str(attempt_config_path),
        'attempt_output_dir': str(attempt_output_dir),
        'run_dir': str(attempt_run_dir),
        'checkpoint': str(ckpt_candidate),
        'metrics': str(metrics_path_candidate),
        'quality_score': quality_score,
        'quality_components': quality_components,
        **quality_metrics,
    })
    print(f'Candidate {attempt} quality score: {quality_score:.6f}')
    print(json.dumps(quality_metrics, indent=2))

    del candidate_model, Z_candidate, pipe_candidate
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_candidate = min(candidate_reports, key=lambda r: r['quality_score'])
selected_run_dir = Path(best_candidate['run_dir'])
selected_config_path = Path(best_candidate['config'])
CONFIG = str(selected_config_path)

if canonical_run_dir.exists():
    shutil.rmtree(canonical_run_dir)
shutil.copytree(selected_run_dir, canonical_run_dir)

with open(Path(OUTPUT_DIR) / f'{run_name}_candidate_report.json', 'w') as f:
    json.dump({'run_name': run_name, 'selected': best_candidate, 'candidates': candidate_reports}, f, indent=2)

selected_quality_score = float(best_candidate['quality_score'])
selected_quality_metrics = best_candidate

print('\nSelected best candidate:')
print(json.dumps(best_candidate, indent=2))
print('Canonical selected run dir:', canonical_run_dir)

In [ ]:
# 6. Load frozen VAE checkpoint and sanity-check training outputs
import os, sys
from pathlib import Path

REPO_DIR = "/content/thesis_hapticAE"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import numpy as np
import torch

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.eval.evaluate import evaluate_reconstruction, print_metrics
from src.eval.visualize import plot_loss_curves, plot_waveform_comparison

config = load_config(CONFIG)
set_seed(config['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_cfg = config['data']

run_name = config.get('run_name', 'vae_default')
run_dir = Path(OUTPUT_DIR) / run_name
ckpt_path = run_dir / 'best_model.pt'
metrics_path = run_dir / 'metrics.npz'
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

model = build_model(config, device)
load_checkpoint(model, str(ckpt_path), device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print('Frozen VAE checkpoint:', ckpt_path)
print('Metrics:', metrics_path, 'exists=', metrics_path.exists())
print('Trainable parameters after freeze:', sum(p.numel() for p in model.parameters() if p.requires_grad))

best_val_loss = None
if metrics_path.exists():
    metrics = np.load(metrics_path)
    best_val_loss = float(np.min(metrics['val_losses']))
    print(f'Best validation loss: {best_val_loss:.6f}')
    plot_loss_curves(metrics['train_losses'].tolist(), metrics['val_losses'].tolist())

In [ ]:
# 7. Reconstruction evaluation before PCA/freeze
# This is a sanity check only. The selected checkpoint remains frozen.
EVAL_N_SAMPLES = 30

data = build_dataloaders(config, DATA_DIR, batch_size=32)
val_loader = data['val_loader']

vae_result = evaluate_reconstruction(model, val_loader, device, n_samples=EVAL_N_SAMPLES)
vae_ratios = [r['recon_std'] / (r['orig_std'] + 1e-8) for r in vae_result['per_sample']]
eval_std_ratio_mean = float(np.mean(vae_ratios))
eval_mse_mean = float(vae_result['reconstruction_summary']['mse_mean'])
eval_spectral_log_l1_mean = float(vae_result['reconstruction_summary']['spectral_log_l1_mean'])

print(f'VAE mean STD ratio: {eval_std_ratio_mean:.1%}')
print_metrics(vae_result)
plot_waveform_comparison(vae_result['x_np'], vae_result['xhat_np'])
# play_ab_comparison intentionally omitted in this minimal freeze notebook.

---
## PCA and Basic Controls

Minimal PCA improvement included here: PCA is fitted only on deterministic VAE `mu`, and the notebook saves latent sample ids plus PCA artifacts for reuse.

In [ ]:
# 8. Extract deterministic mu latents, fit PCA, and save reusable artifacts
import csv
import json
import pickle

from src.pipelines.latent_extraction import extract_latent_vectors
from src.pipelines.pca_control import fit_pca_pipeline

PCA_DIR = Path(OUTPUT_DIR) / 'pca'
PCA_DIR.mkdir(parents=True, exist_ok=True)
N_COMPONENTS = 8

pca_data = build_dataloaders(config, DATA_DIR, batch_size=64, full_dataset=True)
sample_ids = [os.path.relpath(p, DATA_DIR).replace(os.sep, '/') for p in pca_data['wav_files']]

# Deterministic mu is the default PCA input. Do not use sampled z for this first frozen workflow.
Z_mu = extract_latent_vectors(model, pca_data['all_loader'], device).astype(np.float32)
np.save(PCA_DIR / 'Z_mu.npy', Z_mu)
np.save(PCA_DIR / 'Z.npy', Z_mu)  # Compatibility with existing scripts that expect Z.npy.

with open(PCA_DIR / 'sample_ids.json', 'w') as f:
    json.dump(sample_ids, f, indent=2)
with open(PCA_DIR / 'sample_ids.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['row_index', 'sample_id'])
    writer.writerows([[i, sid] for i, sid in enumerate(sample_ids)])

pipe, Z_pca = fit_pca_pipeline(Z_mu, n_components=N_COMPONENTS, save_dir=str(PCA_DIR))
pca = pipe.named_steps['pca']
scaler = pipe.named_steps['scaler']

ranges = []
for axis in range(Z_pca.shape[1]):
    col = Z_pca[:, axis]
    lo, hi = np.percentile(col, [5, 95])
    ranges.append({
        'axis': axis,
        'pc': f'PC{axis + 1}',
        'p5': float(lo),
        'p95': float(hi),
        'min': float(col.min()),
        'max': float(col.max()),
        'mean': float(col.mean()),
        'std': float(col.std()),
    })

np.savez(
    PCA_DIR / 'pca_artifacts.npz',
    components=pca.components_.astype(np.float32),
    explained_variance=pca.explained_variance_.astype(np.float32),
    explained_variance_ratio=pca.explained_variance_ratio_.astype(np.float32),
    pca_mean=pca.mean_.astype(np.float32),
    scaler_mean=scaler.mean_.astype(np.float32),
    scaler_scale=scaler.scale_.astype(np.float32),
    Z_pca=Z_pca.astype(np.float32),
)
with open(PCA_DIR / 'pca_ranges.json', 'w') as f:
    json.dump(ranges, f, indent=2)

print('Saved PCA directory:', PCA_DIR)
print('Latents:', Z_mu.shape)
print('PCA scores:', Z_pca.shape)
pca_coverage = float(pca.explained_variance_ratio_.sum())
print('Total explained variance:', pca_coverage)


In [ ]:
# 9. Build basic controllable PC specification
from src.pipelines.control_spec import build_controls_spec, save_controls_spec, build_controls_table_md

CTRL_DIR = Path(OUTPUT_DIR) / 'controls'
CTRL_DIR.mkdir(parents=True, exist_ok=True)

spec = build_controls_spec(
    pipe,
    model,
    device,
    Z_pca,
    pca.explained_variance_ratio_,
    T=data_cfg['T'],
    sr=data_cfg['sr'],
    n_sweep_steps=11,
)
save_controls_spec(spec, str(CTRL_DIR / 'controls_spec.json'))

controls_table = build_controls_table_md(spec)
with open(CTRL_DIR / 'controls_table.md', 'w', encoding='utf-8') as f:
    f.write(controls_table)

print('Saved controls spec:', CTRL_DIR / 'controls_spec.json')
print('Saved controls table:', CTRL_DIR / 'controls_table.md')
print('\n'.join(controls_table.splitlines()[:20]))

---
## Freeze to Google Drive

This is the stable handoff for later Frozen VAE + PC labeling + LLM control notebooks.

In [ ]:
# 10. Copy best VAE checkpoint, PCA artifacts, and basic controls to Google Drive
from google.colab import drive
from pathlib import Path
import hashlib
import json
import shutil
import time

drive.mount('/content/drive')

FROZEN_ROOT = Path('/content/drive/MyDrive/thesis/frozen_model_outputs')
FROZEN_RUN_DIR = FROZEN_ROOT / run_name
FROZEN_RUN_DIR.mkdir(parents=True, exist_ok=True)
existing_manifest_path = FROZEN_RUN_DIR / 'frozen_manifest.json'

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def copy_file(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return {'path': str(dst), 'sha256': file_sha256(dst)}

current_quality_metrics = {
    'best_val_loss': float(best_val_loss),
    'std_ratio_mean': float(eval_std_ratio_mean),
    'mse_mean': float(eval_mse_mean),
    'spectral_log_l1_mean': float(eval_spectral_log_l1_mean),
    'pca_coverage': float(pca_coverage),
}
current_quality_score, current_quality_components = compute_quality_score(current_quality_metrics)

existing_quality_score = None
existing_best_val_loss = None
if existing_manifest_path.exists():
    with open(existing_manifest_path) as f:
        existing_manifest = json.load(f)
    existing_quality_score = existing_manifest.get('quality_score')
    existing_best_val_loss = existing_manifest.get('best_val_loss')
    print('Existing frozen quality score:', existing_quality_score)
    print('Existing frozen best val loss:', existing_best_val_loss)

should_update = True
if existing_quality_score is not None:
    should_update = current_quality_score < float(existing_quality_score)
elif existing_best_val_loss is not None:
    # Backward-compatible fallback for older manifests that only stored val loss.
    should_update = current_quality_metrics['best_val_loss'] < float(existing_best_val_loss)

if not should_update:
    print(f'Current quality score ({current_quality_score:.6f}) is not better than Drive ({existing_quality_score}).')
    print('Keeping existing frozen model in Drive unchanged:', FROZEN_RUN_DIR)
else:
    files = {}
    files['checkpoint'] = copy_file(ckpt_path, FROZEN_RUN_DIR / 'best_model.pt')
    if metrics_path.exists():
        files['metrics'] = copy_file(metrics_path, FROZEN_RUN_DIR / 'metrics.npz')
    files['config'] = copy_file(Path(CONFIG), FROZEN_RUN_DIR / 'config.yaml')

    frozen_pca_dir = FROZEN_RUN_DIR / 'pca'
    frozen_controls_dir = FROZEN_RUN_DIR / 'controls'
    if frozen_pca_dir.exists():
        shutil.rmtree(frozen_pca_dir)
    if frozen_controls_dir.exists():
        shutil.rmtree(frozen_controls_dir)
    shutil.copytree(PCA_DIR, frozen_pca_dir)
    shutil.copytree(CTRL_DIR, frozen_controls_dir)

    manifest = {
        'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'run_name': run_name,
        'quality_score': current_quality_score,
        'quality_components': current_quality_components,
        'quality_metrics': current_quality_metrics,
        'selection_weights': SELECTION_WEIGHTS,
        'best_val_loss': current_quality_metrics['best_val_loss'],
        'std_ratio_mean': current_quality_metrics['std_ratio_mean'],
        'mse_mean': current_quality_metrics['mse_mean'],
        'spectral_log_l1_mean': current_quality_metrics['spectral_log_l1_mean'],
        'pca_coverage': current_quality_metrics['pca_coverage'],
        'source_output_dir': str(OUTPUT_DIR),
        'frozen_run_dir': str(FROZEN_RUN_DIR),
        'checkpoint_path': str(FROZEN_RUN_DIR / 'best_model.pt'),
        'config_path': str(FROZEN_RUN_DIR / 'config.yaml'),
        'metrics_path': str(FROZEN_RUN_DIR / 'metrics.npz') if (FROZEN_RUN_DIR / 'metrics.npz').exists() else None,
        'pca_dir': str(frozen_pca_dir),
        'pca_pipe_path': str(frozen_pca_dir / 'pca_pipe.pkl'),
        'pca_artifacts_path': str(frozen_pca_dir / 'pca_artifacts.npz'),
        'latent_path': str(frozen_pca_dir / 'Z_mu.npy'),
        'sample_ids_path': str(frozen_pca_dir / 'sample_ids.json'),
        'controls_dir': str(frozen_controls_dir),
        'controls_spec_path': str(frozen_controls_dir / 'controls_spec.json'),
        'controls_table_path': str(frozen_controls_dir / 'controls_table.md'),
        'files': files,
    }

    with open(FROZEN_RUN_DIR / 'frozen_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    with open(FROZEN_ROOT / 'latest_frozen_manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)

    print('Updated frozen run directory:', FROZEN_RUN_DIR)
    print('Quality score saved:', current_quality_score)
    print(json.dumps(current_quality_metrics, indent=2))
    print('Manifest:', FROZEN_RUN_DIR / 'frozen_manifest.json')
    print('Latest manifest:', FROZEN_ROOT / 'latest_frozen_manifest.json')